# HMS v10 — label-prior correction done per model, per-class effects, fold IDs (CPU only)

**Why.** On the hidden test (50 s excerpts, ≥10 votes) a 200 s shortcut model sees the "short recording" cue on every EEG, so its outputs already lean toward the short-excerpt label mix. Multiplying every model by the same factor π(≥10 votes)/π(all) — what we submitted — then corrects that model twice. The fair correction maps each model *from its own operating prior* to the target prior.

**What this notebook does (no test labels, no leaderboard):** it uses the training EEGs that look like the hidden test — no extra real context *and* ≥10 votes (n ≈ 2,650) — as a proxy test set, fold by fold. Priors always come from the other four folds (patients disjoint), exactly as on Kaggle. Corrections compared:

| method | factor applied to calibrated predictions |
|---|---|
| `none` | — |
| `ge10/all` | π(≥10 votes) / π(all) — what we submitted |
| `ge10/noctx` | π(≥10 votes) / π(no-context) — assumes the model acts as if it had learnt the short-excerpt prior |
| `ge10/self` | π(≥10 votes) / mean prediction of the model on the proxy set itself (label-free, per model) |
| `em` | Saerens–Latinne–Decaestecker EM: the test prior is estimated from the model's own predictions, starting from the model's mean training prediction (label-free, per model) |

**Selection rule, fixed before running:** the primary method is the one with the lowest proxy KL averaged over every model and setting. It is then used unchanged for all new hidden-test submissions (notebook `08`, `PRIOR_SHIFT`). The proxy also tests the paper's claim directly: after a per-model correction, is the 200 s original model still better than the 50 s model on test-like EEGs?

Also: per-class KL contributions (which classes the long context and the shortcut act on) and the fold assignment of every training row, for the repository.

**Inputs (Accelerator: None):** the same as the final-statistics notebook (`notebook2e35ba1ac0`): competition data + outputs of `notebook86c832275b` (Version 5), `notebookd13836b6e5`, `notebookbb6726eb71`, `notebook22e15f591d`, `notebook92c60ba812`, `notebookdcb82b4ee1`. Optional: the v10-seeds notebook output (extra seeds of the original 200 s models) — used automatically if attached. ~5–10 min.

In [1]:
CFG = dict(
    DATA_ROOT=None, OUT="/kaggle/working/prior", PRED_GLOBS=["/kaggle/input/**/*_valpred.npy", "/kaggle/input/**/*_oofpred.npy"],
    N_FOLDS=5, SEED=42, B=4000, T_GRID=[round(0.5 + 0.05 * i, 3) for i in range(51)], MIN_N=20,
    CTX_BINS=[-1, 10, 140, 1000], CTX_LABELS=["none (<10 s)", "partial (10-140 s)", "full (>=140 s)"],
    VOTE_GROUPS=[(1, 2, "1-2 votes"), (3, 9, "3-9 votes"), (10, 10_000, ">=10 votes")],
    SEEDS_50=[42, 1042, 2042, 3042, 4042], SEEDS_200=[42, 1042, 2042, 3042], SEEDS_ORIG=[42, 1042, 2042],
)

In [2]:
import os, glob, time, json, warnings
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.model_selection import StratifiedGroupKFold
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 230); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 300)
os.makedirs(CFG["OUT"], exist_ok=True)

def find_root():
    if CFG["DATA_ROOT"]:
        return CFG["DATA_ROOT"]
    for pat in ["/kaggle/input/*/train.csv", "/kaggle/input/*/*/train.csv", "/kaggle/input/*/*/*/train.csv"]:
        for h in glob.glob(pat):
            if os.path.isdir(os.path.join(os.path.dirname(h), "train_eegs")):
                return os.path.dirname(h)
    raise FileNotFoundError("Attach the competition data")
ROOT = find_root()
VOTES = ["seizure_vote", "lpd_vote", "gpd_vote", "lrda_vote", "grda_vote", "other_vote"]
CLASSES = ["Seizure", "LPD", "GPD", "LRDA", "GRDA", "Other"]
P_COLS = [f"p_{c}" for c in CLASSES]
TABLES = {}
def save_table(name, d, show=True):
    TABLES[name] = d
    d.to_csv(f"{CFG['OUT']}/{name}.csv", index=False)
    if show:
        print(f"\n### {name}\n" + d.round(4).to_string(index=False))

## Labels, folds, out-of-fold rows, context availability

In [3]:
df = pd.read_csv(f"{ROOT}/train.csv")
df["n_votes"] = df[VOTES].sum(axis=1)
df[P_COLS] = df[VOTES].div(df["n_votes"], axis=0).to_numpy()
df["y"] = df[P_COLS].to_numpy().argmax(1)
df["center"] = df["eeg_label_offset_seconds"] + 25.0
sgkf = StratifiedGroupKFold(n_splits=CFG["N_FOLDS"], shuffle=True, random_state=CFG["SEED"])
df["fold"] = -1
for k, (_, va) in enumerate(sgkf.split(df, df["y"], groups=df["patient_id"])):
    df.loc[df.index[va], "fold"] = k

def middle_row_per_eeg(rows):
    rows = rows.sort_values(["eeg_id", "eeg_label_offset_seconds"])
    g = rows.groupby("eeg_id")
    return rows[g.cumcount().to_numpy() == (g["eeg_id"].transform("size") // 2).to_numpy()]
VA = {f: middle_row_per_eeg(df[df.fold == f]).reset_index(drop=True) for f in range(CFG["N_FOLDS"])}
OOF = pd.concat([VA[f] for f in range(CFG["N_FOLDS"])], ignore_index=True)

t0 = time.time()
dur = {int(e): pq.ParquetFile(f"{ROOT}/train_eegs/{int(e)}.parquet").metadata.num_rows / 200.0 for e in OOF["eeg_id"].unique()}
print(f"read {len(dur)} EEG lengths in {time.time()-t0:.0f}s")
def extra_context(center, d, ctx=200):
    half = ctx / 2
    left = np.minimum(center, half); right = np.clip(np.minimum(d - center, half), 0, None)
    return left + right - np.minimum(center, 25) - np.clip(np.minimum(d - center, 25), 0, None)
OOF["dur_s"] = OOF["eeg_id"].map(dur)
OOF["extra_s"] = extra_context(OOF["center"].to_numpy(), OOF["dur_s"].to_numpy())
OOF["ctx_bin"] = pd.cut(OOF["extra_s"], CFG["CTX_BINS"], labels=CFG["CTX_LABELS"]).astype(str)
OOF["vote_group"] = ""
for lo, hi, name in CFG["VOTE_GROUPS"]:
    OOF.loc[(OOF.n_votes >= lo) & (OOF.n_votes <= hi), "vote_group"] = name

T = OOF[P_COLS].to_numpy(np.float64)
NV = OOF["n_votes"].to_numpy(); FOLD = OOF["fold"].to_numpy()
CB, VG = OOF["ctx_bin"].to_numpy(), OOF["vote_group"].to_numpy()
ALL = np.ones(len(OOF), bool); HV = NV >= 10
NONE, PART, FULL = (CB == CFG["CTX_LABELS"][0]), (CB == CFG["CTX_LABELS"][1]), (CB == CFG["CTX_LABELS"][2])
REAL = ~NONE
UP, INV = np.unique(OOF["patient_id"].to_numpy(), return_inverse=True)
print(f"{len(OOF)} EEGs, {len(UP)} patients")

read 17089 EEG lengths in 224s
17089 EEGs, 1950 patients


### Table 1 — data: context availability × annotation round, and label priors

In [4]:
ct = pd.crosstab(OOF["vote_group"], OOF["ctx_bin"], margins=True).reset_index()
save_table("T1a_context_by_votes", ct)
pri = {"all EEGs": ALL, "no extra context": NONE, "partial context": PART, "full context": FULL, ">=10 votes": HV, "no context & >=10 votes": NONE & HV}
rows = []
for k, m in pri.items():
    rows.append(dict(subset=k, n=int(m.sum()), median_votes=float(np.median(NV[m])), share_ge10=float(HV[m].mean()),
                     **{c: T[m, i].mean() for i, c in enumerate(CLASSES)}))
save_table("T1b_label_priors", pd.DataFrame(rows))


### T1a_context_by_votes
vote_group  full (>=140 s)  none (<10 s)  partial (10-140 s)   All
 1-2 votes              25          4526                 296  4847
 3-9 votes             297          2298                4047  6642
>=10 votes             169          2648                2783  5600
       All             491          9472                7126 17089

### T1b_label_priors
                 subset     n  median_votes  share_ge10  Seizure    LPD    GPD   LRDA   GRDA  Other
               all EEGs 17089           3.0      0.3277   0.1541 0.1425 0.1043 0.0654 0.1148 0.4189
       no extra context  9472           3.0      0.2796   0.0680 0.1360 0.1205 0.0453 0.0968 0.5334
        partial context  7126           3.0      0.3905   0.2631 0.1484 0.0814 0.0891 0.1308 0.2872
           full context   491           3.0      0.3442   0.2347 0.1808 0.1230 0.1099 0.2297 0.1219
             >=10 votes  5600          15.0      1.0000   0.0425 0.1829 0.1251 0.0740 0.0904 0.4851
no context & >=10

## Saved predictions

In [5]:
FILES = {}
for pat in CFG["PRED_GLOBS"]:
    for fp in glob.glob(pat, recursive=True):
        FILES.setdefault(os.path.basename(fp), fp)
print(f"{len(FILES)} prediction files found")

def pooled(fmt):
    """fmt contains {f}; returns pooled OOF predictions or None"""
    parts = []
    for f in range(CFG["N_FOLDS"]):
        fp = FILES.get(fmt.format(f=f))
        if fp is None:
            return None
        P = np.load(fp); assert len(P) == len(VA[f]), fp
        parts.append(P)
    return np.concatenate(parts).astype(np.float64)

def get(family, setting, seed=42, view="std"):
    """family: mamba/gru/transformer/cnn/mamba_causal/cbramod; setting: 50s1, 50s2, 200orig, 200ba, 200banr; view: std/reflect/truncate/trunc50"""
    sfx = "" if seed == 42 else f"_seed{seed}"
    if family == "cbramod":
        if seed != 42: return None
        if setting == "50s1" and view == "std": return pooled("cbramod_orig_ctx50_f{f}_standard_valpred.npy")
        if setting == "200orig": return pooled("cbramod_orig_ctx200_f{f}_" + {"std": "standard"}.get(view, view) + "_valpred.npy")
        if setting == "200banr": return pooled("cbramod_ba_noreweight_ctx200_f{f}_" + {"std": "ba"}.get(view, view) + "_valpred.npy")
        return None
    if setting in ("50s1", "50s2") and view == "std":
        st = setting[-2:]
        return pooled(f"{family}_ctx50_f{{f}}_{st}_valpred.npy") if seed == 42 else pooled(f"{family}_ctx50_f{{f}}{sfx}_{st}_valpred.npy")
    if setting == "200orig":
        if seed != 42:      # extra seeds of the original recipe (v10 seeds notebook)
            return pooled(f"{family}_ctx200_f{{f}}{sfx}_s1" + ("" if view == "std" else f"_{view}") + "_valpred.npy")
        if view == "std":
            return pooled(f"{family}_ctx200_f{{f}}_s1_valpred.npy")
        fp = FILES.get(f"{family}_ctx200_{view}_oofpred.npy")
        return np.load(fp).astype(np.float64) if fp else None
    if setting in ("200ba", "200banr"):
        var = "ba" if setting == "200ba" else "ba_noreweight"
        return pooled(f"{family}_{var}_ctx200_f{{f}}{sfx}_" + {"std": "ba"}.get(view, view) + "_valpred.npy")
    return None

def per_eeg_kl(P, eps=1e-15):
    return (T * (np.log(T + eps) - np.log(P + eps))).sum(1)
def temper(P, t):
    L = np.log(np.clip(P, 1e-12, 1)) / t; L -= L.max(1, keepdims=True); E = np.exp(L)
    return E / E.sum(1, keepdims=True)
def fit_temps(P, fit_mask):
    Ts = {}
    for f in range(CFG["N_FOLDS"]):
        tr = (FOLD != f) & fit_mask
        Ts[f] = min(CFG["T_GRID"], key=lambda t: (T[tr] * (np.log(T[tr] + 1e-15) - np.log(temper(P[tr], t) + 1e-15))).sum(1).mean())
    return Ts
def apply_temps(P, Ts):
    out = np.empty_like(P)
    for f, t in Ts.items():
        out[FOLD == f] = temper(P[FOLD == f], t)
    return out

_KL, _TS, _RAW = {}, {}, {}
def kl(family, setting, seed=42, view="std"):
    """calibrated per-EEG KL; intervention views reuse the temperatures fitted on the model's own standard view"""
    key = (family, setting, seed, view)
    if key in _KL:
        return _KL[key]
    P = get(family, setting, seed, view)
    if P is None:
        _KL[key] = None; return None
    base = (family, setting, seed)
    if base not in _TS:
        Pstd = P if view == "std" else get(family, setting, seed, "std")
        if Pstd is None:
            _KL[key] = None; return None
        _TS[base] = fit_temps(Pstd, HV if setting == "50s2" else ALL)
    _RAW[key] = per_eeg_kl(P)
    _KL[key] = per_eeg_kl(apply_temps(P, _TS[base]))
    return _KL[key]

# sanity: the shortcut-check standard view reproduces the saved original predictions
for fam in ["mamba", "gru", "transformer", "cnn"]:
    a, b = FILES.get(f"{fam}_ctx200_standard_oofpred.npy"), get(fam, "200orig")
    if a and b is not None:
        print(f"{fam} 200 s original: shortcut-check standard view vs saved predictions, max |dp| = {np.abs(np.load(a) - b).max():.4f}")

425 prediction files found
mamba 200 s original: shortcut-check standard view vs saved predictions, max |dp| = 0.0010
gru 200 s original: shortcut-check standard view vs saved predictions, max |dp| = 0.0009
transformer 200 s original: shortcut-check standard view vs saved predictions, max |dp| = 0.0009
cnn 200 s original: shortcut-check standard view vs saved predictions, max |dp| = 0.0006


## Shared patient-level cluster bootstrap

In [6]:
W = np.random.default_rng(CFG["SEED"]).multinomial(len(UP), np.full(len(UP), 1.0 / len(UP)), size=CFG["B"]).astype(np.float32)
def rep_mean(d, msk):
    """bootstrap replicates of the mean of d over mask (patients resampled)"""
    s_ = np.bincount(INV[msk], weights=d[msk], minlength=len(UP)).astype(np.float32)
    n_ = np.bincount(INV[msk], minlength=len(UP)).astype(np.float32)
    return (W @ s_) / np.maximum(W @ n_, 1e-9)
def summarize(point, reps):
    reps = np.asarray(reps, float)
    if not np.isfinite(point) or not np.isfinite(reps).any():
        return dict(est=np.nan, ci_lo=np.nan, ci_hi=np.nan, p=np.nan)
    reps = reps[np.isfinite(reps)]
    lo, hi = np.percentile(reps, [2.5, 97.5])
    p = max(min(1.0, 2 * min((reps <= 0).mean(), (reps >= 0).mean())), 1.0 / CFG["B"])
    return dict(est=point, ci_lo=lo, ci_hi=hi, p=p)
def diff_test(ka, kb, msk):
    """mean(ka - kb) over mask; relative to mean(kb)"""
    d = ka - kb
    r = summarize(d[msk].mean(), rep_mean(d, msk))
    r["rel_pct"] = 100 * r["est"] / kb[msk].mean()
    r.update(n=int(msk.sum()), kl_a=ka[msk].mean(), kl_b=kb[msk].mean())
    return r
def rel_gain_reps(ka, kb, msk):
    return rep_mean(ka - kb, msk) / rep_mean(kb, msk), (ka - kb)[msk].mean() / kb[msk].mean()
def holm(p):
    p = np.asarray(p, float); out = np.full_like(p, np.nan); ok = np.where(np.isfinite(p))[0]
    o = ok[np.argsort(p[ok])]; run = 0.0
    for i, ix in enumerate(o):
        run = max(run, min(1.0, (len(o) - i) * p[ix])); out[ix] = run
    return out

## Calibrated predictions (cross-fitted temperatures, as in the paper)

In [7]:
_CAL = {}
def cal(fam, setting, seed=42):
    key = (fam, setting, seed)
    if key not in _CAL:
        P = get(fam, setting, seed)
        _CAL[key] = None if P is None else apply_temps(P, fit_temps(P, HV if setting == "50s2" else ALL))
    return _CAL[key]

SETS = [("50s1", CFG["SEEDS_50"]), ("50s2", CFG["SEEDS_50"]), ("200orig", CFG["SEEDS_ORIG"]), ("200banr", CFG["SEEDS_200"])]
FAMS = ["mamba", "gru", "transformer", "cnn"]
HAVE = {(fam, st): [s for s in seeds if cal(fam, st, s) is not None] for fam in FAMS for st, seeds in SETS}
print("seeds found:\n" + "\n".join(f"  {fam:12s} {st:8s} {HAVE[(fam, st)]}" for (fam, st) in HAVE if HAVE[(fam, st)]))

seeds found:
  mamba        50s1     [42, 1042, 2042, 3042, 4042]
  mamba        50s2     [42, 1042, 2042, 3042, 4042]
  mamba        200orig  [42]
  mamba        200banr  [42, 1042, 2042, 3042]
  gru          50s1     [42, 1042, 2042, 3042, 4042]
  gru          50s2     [42, 1042, 2042, 3042, 4042]
  gru          200orig  [42]
  gru          200banr  [42, 1042, 2042, 3042]
  transformer  50s1     [42, 1042, 2042]
  transformer  50s2     [42, 1042, 2042]
  transformer  200orig  [42]
  transformer  200banr  [42, 1042, 2042]
  cnn          50s1     [42, 1042, 2042]
  cnn          50s2     [42, 1042, 2042]
  cnn          200orig  [42]


## Proxy test set and the five corrections

In [8]:
S = NONE & HV
print(f"proxy test set: {S.sum()} EEGs, {len(np.unique(INV[S]))} patients; label prior "
      + ", ".join(f"{c} {v:.3f}" for c, v in zip(CLASSES, T[S].mean(0))))
METHODS = ["none", "ge10/all", "ge10/noctx", "ge10/self", "em"]

def em_adjust(P, pi_tr, iters=1000, tol=1e-9):
    """Saerens et al. (2002): EM estimate of the new prior from posteriors P obtained under prior pi_tr."""
    pi = pi_tr.copy()
    for _ in range(iters):
        Q = P * (pi / pi_tr); Q /= Q.sum(1, keepdims=True)
        new = Q.mean(0)
        done = np.abs(new - pi).max() < tol
        pi = new
        if done:
            break
    Q = P * (pi / pi_tr)
    return Q / Q.sum(1, keepdims=True)

def corrected(P, method, mask=S):
    """P: calibrated pooled OOF predictions (all EEGs). Returns predictions for the EEGs in mask, fold by fold."""
    out = np.empty((mask.sum(), P.shape[1]))
    idx = np.where(mask)[0]
    for f in range(CFG["N_FOLDS"]):
        sel = FOLD[idx] == f
        Pf = P[idx[sel]]
        other = FOLD != f
        pi_all, pi_ge10, pi_noctx = T[other].mean(0), T[other & HV].mean(0), T[other & NONE].mean(0)
        if method == "none":
            Q = Pf
        elif method == "em":
            Q = em_adjust(Pf, P[other].mean(0))
        else:
            R = {"ge10/all": pi_ge10 / pi_all, "ge10/noctx": pi_ge10 / pi_noctx, "ge10/self": pi_ge10 / Pf.mean(0)}[method]
            Q = Pf * R[None, :]
        out[sel] = Q / Q.sum(1, keepdims=True)
    return out

def kl_rows(P, Tm, eps=1e-15):
    return (Tm * (np.log(Tm + eps) - np.log(P + eps))).sum(1)

# per-EEG KL on the proxy set, for every model, setting, seed and method
KS = {}
for (fam, st), seeds in HAVE.items():
    for s in seeds:
        for m in METHODS:
            KS[(fam, st, s, m)] = kl_rows(corrected(cal(fam, st, s), m), T[S])

rows = []
for (fam, st), seeds in HAVE.items():
    if not seeds:
        continue
    for m in METHODS:
        v = np.array([KS[(fam, st, s, m)].mean() for s in seeds])
        rows.append(dict(model=fam, setting=st, method=m, n_seeds=len(seeds), kl_mean=v.mean(), kl_sd=v.std(ddof=1) if len(v) > 1 else np.nan))
proxy = pd.DataFrame(rows)
save_table("T10a_proxy_kl", proxy, show=False)
print(proxy.pivot_table(index=["setting", "model"], columns="method", values="kl_mean", sort=False)[METHODS].round(4).to_string())

proxy test set: 2648 EEGs, 698 patients; label prior Seizure 0.042, LPD 0.140, GPD 0.125, LRDA 0.048, GRDA 0.075, Other 0.569
method                 none  ge10/all  ge10/noctx  ge10/self      em
setting model                                                       
50s1    mamba        0.4918    0.4126      0.4831     0.4256  0.9572
        gru          0.5007    0.4212      0.4916     0.4303  0.9171
        transformer  0.5320    0.4357      0.5148     0.4431  1.0701
        cnn          0.5242    0.4484      0.5208     0.4564  0.8702
50s2    mamba        0.3700    0.3876      0.3974     0.3856  1.9591
        gru          0.3717    0.3924      0.4004     0.3880  1.9948
        transformer  0.3923    0.4074      0.4189     0.4058  1.8752
        cnn          0.3992    0.4190      0.4328     0.4136  1.8125
200orig mamba        0.4277    0.4010      0.4348     0.4189  1.1981
        gru          0.4456    0.4204      0.4514     0.4337  2.0241
        transformer  0.4631    0.4357      0.4

## Primary method (rule fixed above) and the shortcut claim under each method
Comparisons use per-EEG KL averaged over the seeds both settings share (seed 42 for single-seed settings), patient-level cluster bootstrap on the proxy set, Holm within each method.

In [9]:
rank = proxy.groupby("method")["kl_mean"].mean().reindex(METHODS)
PRIMARY = rank.idxmin()
print("mean proxy KL over all models and settings:\n" + rank.round(4).to_string())
print(f"\nPRIMARY correction (selection rule): {PRIMARY}")

def seed_mean(fam, st, m, seeds):
    return np.mean([KS[(fam, st, s, m)] for s in seeds], 0)
rows = []
for m in METHODS:
    for fam in FAMS:
        for a, b in [("200orig", "50s1"), ("200banr", "50s1"), ("200orig", "200banr"), ("50s2", "50s1")]:
            sa, sb = HAVE.get((fam, a), []), HAVE.get((fam, b), [])
            if not sa or not sb:
                continue
            common = [s for s in sa if s in sb] or None
            ka = seed_mean(fam, a, m, common or sa); kb = seed_mean(fam, b, m, common or sb)
            d = np.zeros(len(OOF)); d[S] = ka - kb
            r = summarize(d[S].mean(), rep_mean(d, S))
            rows.append(dict(method=m, model=fam, comparison=f"{a} - {b}", seeds=("shared " + str(common)) if common else f"{sa} vs {sb}",
                             kl_a=ka.mean(), kl_b=kb.mean(), diff=r["est"], ci_lo=r["ci_lo"], ci_hi=r["ci_hi"], rel_pct=100 * r["est"] / kb.mean(), p=r["p"]))
cmp_ = pd.DataFrame(rows)
cmp_["p_holm_method"] = cmp_.groupby("method")["p"].transform(lambda s: pd.Series(holm(s.to_numpy()), index=s.index))
save_table("T10b_proxy_comparisons", cmp_, show=False)
for m in METHODS:
    print(f"\n### method {m}" + ("   <-- primary" if m == PRIMARY else ""))
    print(cmp_[cmp_.method == m].drop(columns=["method"]).round(4).to_string(index=False))

# mean corrected prediction vs the proxy set's true prior (Seizure and Other)
rows = []
for fam in FAMS:
    for st in ["50s1", "200orig", "200banr"]:
        if HAVE.get((fam, st)):
            for m in METHODS:
                mp_ = corrected(cal(fam, st, HAVE[(fam, st)][0]), m).mean(0)
                rows.append(dict(model=fam, setting=st, method=m, **{f"mean_{c}": mp_[i] for i, c in enumerate(CLASSES)}))
save_table("T10c_mean_prediction_on_proxy", pd.DataFrame(rows), show=False)
print("\ntrue proxy prior: " + ", ".join(f"{c} {v:.3f}" for c, v in zip(CLASSES, T[S].mean(0))))
print(TABLES["T10c_mean_prediction_on_proxy"].pivot_table(index=["setting", "model"], columns="method", values="mean_Seizure", sort=False)[METHODS].round(3).to_string())
json.dump(dict(primary=PRIMARY, mean_proxy_kl=rank.round(5).to_dict()), open(f"{CFG['OUT']}/primary_method.json", "w"), indent=1)

mean proxy KL over all models and settings:
method
none          0.4583
ge10/all      0.4189
ge10/noctx    0.4636
ge10/self     0.4266
em            1.5010

PRIMARY correction (selection rule): ge10/all

### method none
      model        comparison                               seeds   kl_a   kl_b    diff   ci_lo   ci_hi  rel_pct      p  p_holm_method
      mamba    200orig - 50s1                         shared [42] 0.4277 0.4949 -0.0672 -0.0855 -0.0489 -13.5768 0.0002         0.0035
      mamba    200banr - 50s1       shared [42, 1042, 2042, 3042] 0.4784 0.4934 -0.0149 -0.0257 -0.0042  -3.0252 0.0060         0.0120
      mamba 200orig - 200banr                         shared [42] 0.4277 0.4860 -0.0582 -0.0758 -0.0417 -11.9829 0.0002         0.0035
      mamba       50s2 - 50s1 shared [42, 1042, 2042, 3042, 4042] 0.3700 0.4918 -0.1218 -0.1416 -0.1049 -24.7648 0.0002         0.0035
        gru    200orig - 50s1                         shared [42] 0.4456 0.5063 -0.0606 -0.0771 -0.0449 -

## Per-class effects
KL decomposes over classes: KL = Σ_c t_c log(t_c / p_c). The table gives each class's contribution (seed-averaged, calibrated) and, for the comparisons that matter, the per-class difference with a patient-level interval.

In [10]:
def per_class(P, eps=1e-15):
    return T * (np.log(T + eps) - np.log(P + eps))            # (n, 6); rows sum to the per-EEG KL
PC = {}
for fam in FAMS:
    for st in ["50s1", "50s2", "200orig", "200banr"]:
        seeds = HAVE.get((fam, st))
        if seeds:
            PC[(fam, st)] = np.mean([per_class(cal(fam, st, s)) for s in seeds], 0)
rows = []
for (fam, st), C in PC.items():
    for sname, msk in [("all EEGs", ALL), (">=10 votes", HV), ("no context", NONE), ("with real context", REAL)]:
        rows.append(dict(model=fam, setting=st, subset=sname, n=int(msk.sum()), total=C[msk].sum(1).mean(),
                         **{c: C[msk, i].mean() for i, c in enumerate(CLASSES)}))
save_table("T11a_per_class_kl", pd.DataFrame(rows), show=False)
rows = []
for fam in FAMS:
    for a, b in [("200banr", "50s1"), ("200orig", "50s1"), ("200orig", "200banr")]:
        if (fam, a) not in PC or (fam, b) not in PC:
            continue
        D = PC[(fam, a)] - PC[(fam, b)]
        for sname, msk in [("all EEGs", ALL), ("no context", NONE), ("with real context", REAL)]:
            for i, c in enumerate(CLASSES):
                r = summarize(D[msk, i].mean(), rep_mean(D[:, i], msk))
                rows.append(dict(model=fam, comparison=f"{a} - {b}", subset=sname, cls=c, diff=r["est"], ci_lo=r["ci_lo"], ci_hi=r["ci_hi"], p=r["p"]))
pcd = pd.DataFrame(rows)
pcd["p_holm"] = holm(pcd["p"].to_numpy())
save_table("T11b_per_class_differences", pcd, show=False)
print(pcd.pivot_table(index=["model", "comparison", "subset"], columns="cls", values="diff", sort=False)[CLASSES].round(4).to_string())

cls                                              Seizure     LPD     GPD    LRDA    GRDA   Other
model       comparison        subset                                                            
mamba       200banr - 50s1    all EEGs           -0.0091 -0.0078 -0.0024 -0.0026 -0.0007 -0.0032
                              no context          0.0003 -0.0068 -0.0028 -0.0024  0.0003 -0.0070
                              with real context  -0.0207 -0.0089 -0.0019 -0.0028 -0.0018  0.0015
            200orig - 50s1    all EEGs           -0.0215 -0.0053 -0.0018 -0.0038 -0.0107 -0.0141
                              no context          0.0307 -0.0064 -0.0039  0.0014 -0.0007 -0.0797
                              with real context  -0.0863 -0.0039  0.0008 -0.0104 -0.0231  0.0675
            200orig - 200banr all EEGs           -0.0124  0.0024  0.0006 -0.0012 -0.0100 -0.0109
                              no context          0.0304  0.0004 -0.0011  0.0039 -0.0010 -0.0727
                              

## Fold assignment of every training row (for the repository)

In [11]:
df[["eeg_id", "eeg_sub_id", "label_id", "patient_id", "fold"]].to_csv(f"{CFG['OUT']}/fold_assignments.csv", index=False)
OOF[["eeg_id", "patient_id", "fold", "n_votes", "dur_s", "extra_s", "ctx_bin", "vote_group"]].to_csv(f"{CFG['OUT']}/oof_eegs.csv", index=False)
with pd.ExcelWriter(f"{CFG['OUT']}/hms_prior_perclass.xlsx") as xw:
    for k, d in TABLES.items():
        d.to_excel(xw, sheet_name=k[:31], index=False)
print("written:", sorted(os.listdir(CFG["OUT"])))

written: ['T10a_proxy_kl.csv', 'T10b_proxy_comparisons.csv', 'T10c_mean_prediction_on_proxy.csv', 'T11a_per_class_kl.csv', 'T11b_per_class_differences.csv', 'T1a_context_by_votes.csv', 'T1b_label_priors.csv', 'fold_assignments.csv', 'hms_prior_perclass.xlsx', 'oof_eegs.csv', 'primary_method.json']
